Touching price dataset...

# EDA

In [4]:
from pathlib import Path

import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jacksaleeby/s-and-p500-historical-data")

print("Path to dataset files:", path)

/Users/konstantinmelnikov/Desktop/work/portfolio optimization/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/konstantinmelnikov/Desktop/work/portfolio optimization/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 39.5M/39.5M [01:01<00:00, 669kB/s] 

Extracting files...
Path to dataset files: /Users/konstantinmelnikov/.cache/kagglehub/datasets/jacksaleeby/s-and-p500-historical-data/versions/1


In [6]:
dataset_path = Path(path) / "SP500_Historical_Data.csv"
dataset = pd.read_csv(dataset_path)

print(f"Dataset shape: {dataset.shape[0]:,} rows × {dataset.shape[1]} columns")

dataset

Dataset shape: 2,703,531 rows × 8 columns


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
1,A,2000-01-04,40.72,41.17,38.70,39.75,39.75,4765083
2,A,2000-01-05,39.60,39.75,36.05,37.28,37.28,5758642
3,A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
4,A,2000-01-07,35.30,39.41,35.27,38.85,38.85,2819626
...,...,...,...,...,...,...,...,...
2703526,ZS,2026-02-13,174.34,179.90,172.43,177.72,177.72,2434100
2703527,ZS,2026-02-17,175.82,176.85,169.61,172.59,172.59,2038100
2703528,ZS,2026-02-18,167.35,172.65,164.39,172.13,172.13,2542100
2703529,ZS,2026-02-19,169.64,172.10,166.30,168.99,168.99,3337100


In [20]:
dataset = dataset[["Ticker", "Date", "Adj Close"]].copy()

Company analysis

In [31]:
records_per_company = dataset.groupby("Ticker").size()
records_per_company.describe()

count     472.000000
mean     5727.819915
std      1570.555355
min       256.000000
25%      5432.000000
50%      6573.000000
75%      6573.000000
max      6573.000000
dtype: float64

In [35]:
n_companies = 50

dataset = dataset.assign(Date=pd.to_datetime(dataset["Date"]))

company_date_ranges = (
    dataset.groupby("Ticker")
    .agg(
        records=("Date", "size"),
        start_date=("Date", "min"),
        end_date=("Date", "max"),
    )
    .reset_index()
    .sort_values(["records", "Ticker"], ascending=[False, True])
)

selected_companies = company_date_ranges.head(n_companies).reset_index(drop=True)
selected_tickers = selected_companies["Ticker"]
selected_dataset = dataset[dataset["Ticker"].isin(selected_tickers)].copy()

selected_dataset_summary = pd.Series({
    "min start date": selected_companies["start_date"].min(),
    "max start date": selected_companies["start_date"].max(),
    "min end date": selected_companies["end_date"].min(),
    "max end date": selected_companies["end_date"].max(),
    "companies": selected_companies["Ticker"].nunique(),
    "records": len(selected_dataset),
}, name="Значение").to_frame()

selected_dataset_summary

,Значение
min start date,2000-01-03 00:00:00
max start date,2000-01-03 00:00:00
min end date,2026-02-20 00:00:00
max end date,2026-02-20 00:00:00
companies,50
records,328650


# train/val/test split

Initial train period - 2000-2016 
Validation period - 2017-2020
Final test - 2021-2026

In [36]:
train_dataset = selected_dataset.loc[
    selected_dataset["Date"].between("2000-01-01", "2016-12-31")
].copy()

val_dataset = selected_dataset.loc[
    selected_dataset["Date"].between("2017-01-01", "2020-12-31")
].copy()

test_dataset = selected_dataset.loc[
    selected_dataset["Date"].between("2021-01-01", "2026-12-31")
].copy()

splits = {
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset,
}

data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

for split_name, split_dataset in splits.items():
    split_dataset.sort_values(["Ticker", "Date"]).to_csv(
        data_dir / f"{split_name}.csv", index=False
    )

split_summary = pd.DataFrame({
    split_name: {
        "start_date": split_dataset["Date"].min(),
        "end_date": split_dataset["Date"].max(),
        "companies": split_dataset["Ticker"].nunique(),
        "records": len(split_dataset),
        "file": str(data_dir / f"{split_name}.csv"),
    }
    for split_name, split_dataset in splits.items()
}).T

split_summary

,start_date,end_date,companies,records,file
train,2000-01-03 00:00:00,2016-12-30 00:00:00,50,213850,data/train.csv
val,2017-01-03 00:00:00,2020-12-31 00:00:00,50,50350,data/val.csv
test,2021-01-04 00:00:00,2026-02-20 00:00:00,50,64450,data/test.csv
